# LLM-Based Attribute Selection

Reads the attribute CSV and uses the LLM to select the top 10 most relevant attributes for a CDT based on shopper decision relevance.

**Usage:**
1. Place input CSVs in `attribute_selection_data/input/`
2. Run all cells

In [0]:
# Standard library import for parsing LLM JSON responses.
import os
import json

# Data handling for input/output CSV processing.
import pandas as pd

# LangChain message wrappers used to build system and user prompts.
from langchain_core.messages import HumanMessage, SystemMessage

# Azure OpenAI chat client used to call the hosted LLM endpoint.
from langchain_openai import AzureChatOpenAI

## CDTSelector Class

In [0]:
class CDTSelector:
    """Single-stage LLM pipeline: select exactly 10 CDT-relevant attributes."""

    SELECT_PROMPT = """You are selecting attributes for a Customer Decision Tree (CDT).

From the provided list, select EXACTLY 10 attributes that are most relevant to how shoppers make purchase decisions in this category.

Select if:
- It is a real shopper decision driver (e.g., flavor, size, brand tier, form)
- Top Values are meaningful, category-relevant, and differentiating
- It has strong SKU/sales coverage

Do NOT select if:
- Identifier/system field (UPC, item codes)
- Internal classification (e.g., BC*)
- Top Values are mostly NOT STATED, NOT COLLECTED, missing, or non-differentiating
- Noisy/irrelevant to shopper decisions
- Packaging type (box/carded/sleeve) unless packaging IS the product

Value validation (mandatory):
- Inspect Top Values for every attribute
- Judge by dominant high-sales/high-SKU values
- Minor cross-category noise is acceptable if dominant values are relevant
- Heavily penalize attributes where most values are missing or meaningless

Selection rules:
- Avoid redundancy/overlap: never select multiple attributes for the same concept
- Prefer the cleaner attribute when two represent similar ideas
- Ensure diversity across different shopper decision dimensions

Return ONLY JSON:
[
  {"id": <ATTRIBUTE_ID>, "reason": "<short reason>"}
]

Constraints:
- EXACTLY 10 attributes
- reason <= 10 words
- Do NOT rank them; order does not matter
"""

    def __init__(
        self,
        endpoint: str = "https://cis-rnd-llm-api.cis.nielseniq.com/",
        api_secret: str = "cf5f1885-5215-48a3-ad80-c77b0901f3f7",
        api_version: str = "2024-10-21",
        api_key: str = "Nokey",
        model: str = "AAC-gpt-41",
    ):
        self.llm = AzureChatOpenAI(
            azure_endpoint=endpoint,
            api_version=api_version,
            api_key=api_key,
            model=model,
            temperature=0,
            max_tokens=None,
            timeout=None,
            max_retries=2,
            default_headers={"x-niq-cis-consumer": api_secret},
        )

    def _build_select_prompt(self, df: pd.DataFrame) -> str:
        """Build the user prompt presenting all attributes for selection."""
        category_name = df["CATEGORY_NAME"].iloc[0]
        prompt = f"Category: {category_name}\n\n"
        prompt += "Select EXACTLY 10 attributes from the list below.\n\n"

        for _, row in df.iterrows():
            prompt += (
                f"- ATTRIBUTE_ID: {int(row['ATTRIBUTE_ID'])}\n"
                f"  Name: {row['ATTRIBUTE_DESCRIPTION']}\n"
                f"  SKU Count: {row['ATTRIBUTE_COVERAGE_SKU_COUNT']}\n"
                f"  Total Sales: ${row['ATTRIBUTE_TOTAL_SPENT']:,.0f}\n"
            )
            if pd.notna(row.get("TOP_VALUES")):
                prompt += f"  Top Values: {row['TOP_VALUES']}\n"
            prompt += "\n"

        prompt += "Return JSON array with EXACTLY 10 selected attributes."
        return prompt

    def _parse_select_response(self, response_text: str, valid_ids: set) -> list:
        """Parse the LLM selection response. Returns list of parsed items."""
        try:
            start = response_text.index("[")
            end = response_text.rindex("]") + 1
            items = json.loads(response_text[start:end])

            parsed_items = []
            seen_ids = set()

            for item in items:
                try:
                    aid = int(item["id"])
                except (KeyError, TypeError, ValueError):
                    continue

                if aid in seen_ids or aid not in valid_ids:
                    continue

                parsed_items.append({
                    "id": aid,
                    "reason": str(item.get("reason", "")),
                })
                seen_ids.add(aid)

            return parsed_items

        except (ValueError, json.JSONDecodeError):
            print("Warning: Failed to parse selection response.")
            print(response_text)
            return []

    def select(self, input_csv: str, output_csv: str = None, max_retries: int = 3) -> pd.DataFrame:
        """Select exactly 10 CDT-relevant attributes via LLM."""
        df = pd.read_csv(input_csv)
        df = df.dropna(subset=["ATTRIBUTE_ID"])
        valid_ids = set(df["ATTRIBUTE_ID"].astype(int))
        print(f"Loaded {len(df)} attributes | Category: {df['CATEGORY_NAME'].iloc[0]}")

        messages = [
            SystemMessage(content=self.SELECT_PROMPT),
            HumanMessage(content=self._build_select_prompt(df)),
        ]

        selected_items = []
        for attempt in range(1, max_retries + 1):
            response = self.llm.invoke(messages)
            print(f"Response received (attempt {attempt}/{max_retries}).")
            selected_items = self._parse_select_response(response.content, valid_ids)
            if len(selected_items) == 10:
                break
            correction = (
                f"You returned {len(selected_items)} valid attributes instead of 10. "
                f"Return EXACTLY 10 valid ATTRIBUTE_IDs from the list provided."
            )
            print(f"Attempt {attempt}: {correction}")
            messages.append(response)
            messages.append(HumanMessage(content=correction))

        if len(selected_items) != 10:
            raise RuntimeError(
                f"LLM returned {len(selected_items)} valid attributes after {max_retries} attempts. Expected exactly 10."
            )

        # Build output DataFrame.
        selected_ids = [item["id"] for item in selected_items]
        id_to_reason = {item["id"]: item["reason"] for item in selected_items}

        result_df = df[df["ATTRIBUTE_ID"].isin(selected_ids)].copy()
        result_df["REASONING"] = result_df["ATTRIBUTE_ID"].map(id_to_reason)
        result_df = result_df.reset_index(drop=True)

        output_cols = ["ATTRIBUTE_DESCRIPTION", "REASONING", "ATTRIBUTE_TOTAL_SPENT", "TOP_VALUES"]

        if output_csv:
            result_df[output_cols].to_csv(output_csv, index=False)
            print(f"\nSaved to: {output_csv}")

        return result_df[output_cols]

## Run Selection

In [0]:
# Create selector instance with default Azure endpoint/model configuration.
selector = CDTSelector()

# Run the selection pipeline: LLM picks exactly 10 relevant attributes.
result = selector.select(
    r"C:\AJ\LLM\attribute_selection_data\input\pizza_input.csv",
    output_csv=r"C:\AJ\LLM\attribute_selection_data\output\pizza_output.csv",
)

In [0]:
# Display selected attributes and LLM reasoning.
result